In [ ]:
import os
from functools import partial
import pandas as pd
import numpy as np

from madrigal.evaluate.predict import get_drugbank_scores_wrapper
from madrigal.utils import BASE_DIR

# Utils

In [ ]:
drug_metadata = pd.read_pickle(os.path.join(BASE_DIR, 'processed_data/drug_features/drug_metadata_ddi.pkl'))
drug_metadata['view_str'] = 1
print(drug_metadata.shape[0])

drugbank_ddi_classes = pd.read_pickle(BASE_DIR + "processed_data/drug_combination_data/DrugBank/drugbank_ddi_directed_label_map.pkl")
drugbank_ddi_df = pd.read_csv(BASE_DIR + "processed_data/drug_combination_data/DrugBank/drugbank_ddi_directed.tsv", index_col=0)
get_drugbank_scores = partial(get_drugbank_scores_wrapper, ckpt_list=['all_train_seed1', 'all_train_seed0', 'all_train_seed99', 'all_train_seed42', 'all_train_seed2'], drugbank_ddi_classes=drugbank_ddi_classes)

21842


# Make predictions

In [3]:
combos = [
    ["Doxycycline", "Digoxin"],
    ["Doxycycline", "Warfarin"],
    ["Doxycycline", "Tacrolimus"],
    ["Doxycycline", "Levetiracetam"],
    ["Doxycycline", "Piracetam"],
]
outcomes = [
    'excretion rate, decrease | serum level, increase',
    'excretion, decrease',
    'serum level of the active metabolites, increase',
    'serum level, increase',
]

In [4]:
drug_names = np.unique(combos)
drug_inds = [drug_metadata[drug_metadata["node_name"] == drug_name].index.values[0] for drug_name in drug_names]

drug_1_ind_inds = [drug_names.tolist().index(pair[0]) for pair in combos]
drug_2_ind_inds = [drug_names.tolist().index(pair[1]) for pair in combos]

In [21]:
import warnings
import contextlib
import io
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with contextlib.redirect_stdout(io.StringIO()):
        drugbank_scores = get_drugbank_scores(
            outcome_drugbank_inds=[drugbank_ddi_classes.tolist().index(o) for o in outcomes], 
            drug_inds=drug_inds, 
        )

## Predicted scores

In [ ]:
pd.DataFrame(drugbank_scores[:, drug_1_ind_inds, drug_2_ind_inds], index=outcomes, columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates()

## Prediction scores
Prediction scores rank a score among those of all DrugBank drug pairs for the same outcome. They are read from the precomputed file listed in README.md.

In [ ]:
# Full normalized-rank tensor (80 GB); see README.md for the download link.
normalized_rank_drugbank = np.load(BASE_DIR + "model_output/DrugBank/split_by_pairs/DrugBank_drugs_normalized_ranks.npy", mmap_mode="r")
pd.DataFrame(normalized_rank_drugbank[[drugbank_ddi_classes.tolist().index(o) for o in outcomes], :, :][:, np.array(drug_inds)[drug_1_ind_inds], np.array(drug_inds)[drug_2_ind_inds]], index=outcomes, columns=list(zip(np.array(drug_names)[drug_1_ind_inds], np.array(drug_names)[drug_2_ind_inds]))).drop_duplicates()